In [1]:
import torch
import torch.nn as nn

class Linear(nn.Module):
    def __init__(self,in_features,out_features,device=None,dtype=None):
        super().__init__()
        '''
        从0实现一继承自torch.nn.Module 的自定义的Linear类 y=xW^T 无bias
        int_features: d_in
        out_features: d_out
        nn.Parameter()定义权重参数，自动追踪梯度 注意这里的维度是(out,int)所以需要转置
        权重使用截断正态分布 nn.init.trunc_normal_ 初始化,mean=0,std=std,截断范围[-3*std,3*std]
        '''
        self.w=nn.Parameter(torch.empty(out_features,in_features,device=device,dtype=dtype))
        mean,std=0,(2/(in_features+out_features))**0.5
        nn.init.trunc_normal_(self.w,mean,std=std,a=-3*std,b=3*std)
        
    def forward(self,x):
        y=x @ self.w.T
        return y



In [2]:
m=Linear(256,512)
input=torch.randn(32,100,256)
output=m(input)
print(f"output's shape {output.shape}")


output's shape torch.Size([32, 100, 512])


In [3]:
#Embedding 
#(Batch_size,seq_len)  --->  (batch_size,seq_len,emb_dim)
'''
vocab_size词表大小,
embed_dim每个词用多少维度的向量嵌入表示
权重使用截断正态分布 nn.init.trunc_normal_ 初始化 ，均值 0、标准差 1,截断范围为 [-3, 3]
'''
class Embedding(nn.Module):
    def __init__(self,vocab_size,embed_dim,device=None,dtype=None):
        super().__init__()
        self.embed=nn.Parameter(torch.empty(vocab_size,embed_dim,device=device,dtype=dtype))
        nn.init.trunc_normal_(self.embed,mean=0,std=1,a=-3,b=3)
    
    def forward(self,token_id):
        return self.embed[token_id]

In [4]:
emb=Embedding(1000,512)
token_id=torch.tensor([[1,2,3],[4,5,6]]) #(2,3)
output=emb(token_id)
print(output.shape)

torch.Size([2, 3, 512])


In [ ]:
'''
Problem 3: 实现 RMSNorm (1 point)
要求：从零实现一个 RMSNorm 类，对输入最后一维做均方根归一化。接收的核心初始化参数有：
- embed_dim: int — 嵌入维度
- eps: float = 1e-5 — 数值稳定性常数，防止除以零
- device / dtype — 设备与数据类型
实现时需注意数值稳定性：计算过程中转为 float32 精度，计算完成后转回原精度。
为提高 GPU 利用率，模型权重通常采用更低精度的float16/bfloat16 存储；计算时转 float32 防止低精度下出现计算溢出、精度损失等问题
'''

In [5]:
class RMSNorm(nn.Module):
    def __init__(self,embed_dim,eps=1e-5,device=None,dtype=None):
        super().__init__()
        self.embed_dim=embed_dim
        self.eps=eps
        self.g=nn.Parameter(torch.ones(embed_dim,device=device,dtype=dtype)) #(embed_dim)
    
    def forward(self,x):
        # [...,embed_dim]
        #记录原始精度
        in_dtype=x.dtype
        x=x.to(torch.float32)

        #求均方根--->归一化--->*可学习缩放系数
        #(...,embed_dim)-->(...,1)
        rms=torch.sqrt(x.square().sum(dim=-1,keepdim=True)/self.embed_dim+self.eps) #求平均值+eps
        result=x/rms*self.g #(...,embed_dim)/(...,1) *(embed_dim) 右对齐乘embed_dim

        return result.to(in_dtype)
    


In [6]:
x=torch.tensor([
    [1.0,2.0,5.0],
    [-10.0,38.0,23.5],
    [3.9,-17.0,23.2]])

norm=RMSNorm(embed_dim=3)
output=norm(x)

print(f"x:{x}")

print(f"output:{output}")


x:tensor([[  1.0000,   2.0000,   5.0000],
        [-10.0000,  38.0000,  23.5000],
        [  3.9000, -17.0000,  23.2000]])
output:tensor([[ 0.3162,  0.6325,  1.5811],
        [-0.3783,  1.4375,  0.8890],
        [ 0.2327, -1.0145,  1.3844]], grad_fn=<MulBackward0>)


旋转位置编码
Transformer 模型本身无法感知到 token 间的位置关系，两个 token 的距离远近不会影响它们的注意力分数。因此，需要引入位置编码。

In [ ]:
''' 
Problem 4: 实现 RoPE (2 points)
要求：从零实现旋转位置编码模块，将词向量里的每对相邻元素视为一个 2D 平面，按位置旋转角度。接收的核心初始化参数有：
- theta: float — 基数频率（GPT-2 使用 Theta = 10000）
- d_k: int — 待编码的向量维度（一般指多头注意力层里每个头的维度）
- max_seq_len: int — 预计算查找表的最大长度
在代码实现上，无需显式构造完整的 d×d 旋转矩阵，可利用多个二维子矩阵独立完成各组的旋转变换。
各位置对应的 sin 和 cos 值可在不同网络层、不同批次间复用，所有层可以共用同一个 RoPE 模块。
在 RoPE 模块初始化阶段预计算好固定的 sin/cos 值，并使用register_buffer 存储在缓冲区里。
'''

In [7]:
%pip install einops


Note: you may need to restart the kernel to use updated packages.


In [8]:
from einops import rearrange

class Rope(nn.Module):
    def __init__(self, theta,d_k,max_seq_len,device=None):
        super().__init__()

        pos=torch.arange(max_seq_len,device=device)  #(max_seq_len,)
        dim_indices=torch.arange(0,d_k,2,device=device) #(d_k//2,)

        freqs=theta ** -(dim_indices/d_k)

        angle=pos[:,None] * freqs[None,:]

        cos_table=torch.cos(angle)
        sin_table=torch.sin(angle)

        self.register_buffer("cos_table",cos_table)
        self.register_buffer("sin_table",sin_table)

    
    def forward(self,x,token_pos):
        # x->(... seq_len,dk)
        #token_pos->(... seq_len)

        cos=self.cos_table[token_pos]
        sin=self.sin_table[token_pos]

        x1,x2=rearrange(x,"... (a b) -> ... a b",b=2).unbind(dim=-1)

        x1_rot=x1*cos-x2*sin
        x2_rot=x1*sin+x2*cos

        x_rot=torch.stack([x1_rot,x2_rot],dim=-1)
        x_rot=rearrange(x_rot,"... a b -> ... (a b)")

        return x_rot



In [9]:
x=torch.tensor([
    [
        [1,2,3,4],
        [5,6,7,8],
        [9,198,23,43],
        [-20,-43,56,89],
    ],
    [
        [32,43,12,90],
        [-20,98,76,0],
        [1,3,5,3],
        [2,5,7,8]
    ]
]).float()
print(x.shape)

token_pos=torch.tensor([
    [0,1,2,3],
    [4,5,6,7]
])
rope=Rope(theta=10000.0,d_k=4,max_seq_len=1024)
output=rope(x,token_pos)

print(f"输入张量x: {x}")

print(f"输出张量: {output}")

print(output.shape)


torch.Size([2, 4, 4])
输入张量x: tensor([[[  1.,   2.,   3.,   4.],
         [  5.,   6.,   7.,   8.],
         [  9., 198.,  23.,  43.],
         [-20., -43.,  56.,  89.]],

        [[ 32.,  43.,  12.,  90.],
         [-20.,  98.,  76.,   0.],
         [  1.,   3.,   5.,   3.],
         [  2.,   5.,   7.,   8.]]])
输出张量: tensor([[[   1.0000,    2.0000,    3.0000,    4.0000],
         [  -2.3473,    7.4492,    6.9197,    8.0696],
         [-183.7862,  -74.2134,   22.1355,   43.4514],
         [  25.8680,   39.7473,   53.3052,   90.6397]],

        [[  11.6259,  -52.3244,    8.3914,   90.4079],
         [  88.3013,   46.9774,   75.9050,    3.7984],
         [   1.7984,    2.6011,    4.8111,    3.2944],
         [  -1.7771,    5.0835,    6.4233,    8.4700]]])
torch.Size([2, 4, 4])


In [10]:
'''
Problem 5: 实现 softmax (1 point)
要求：实现一个 softmax 函数，对输入张量 x 的第 i 维计算 softmax。
数值稳定性技巧：实现时先减去该维度的最大值，防止指数运算溢出。
'''

'\nProblem 5: 实现 softmax (1 point)\n要求：实现一个 softmax 函数，对输入张量 x 的第 i 维计算 softmax。\n数值稳定性技巧：实现时先减去该维度的最大值，防止指数运算溢出。\n'

In [11]:
def softmax(x: torch.Tensor, i: int) -> torch.Tensor:
    '''
    x: 输入张量，支持任意维度
    i: int  待计算 softmax 的维度编号
    返回形状: 与输入 x 形状完全一致
    '''
    #x(... d)-->(... 1)
    x-=x.max(dim=i,keepdim=True)[0]  #.max()返回一个元组(最大值张量，最大值下标张量)

    return torch.exp(x)/torch.exp(x).sum(dim=i,keepdim=True)

In [12]:
x=torch.tensor([
    [1,2,3],
    [0,0,0]
]).float()

output=softmax(x,i=-1)

print(x)

print(output)

print(output.sum(dim=-1))

tensor([[-2., -1.,  0.],
        [ 0.,  0.,  0.]])
tensor([[0.0900, 0.2447, 0.6652],
        [0.3333, 0.3333, 0.3333]])
tensor([1., 1.])


In [13]:
'''
Problem 6: 实现缩放点积注意力 (5 points)
要求：实现缩放点积注意力，支持任意 batch 维度和可选的注意力掩码。核心参数有：
- q, k: (batch_size, ..., seq_len, d_k) — 查询 query 和键 key
- v: (batch_size, ..., seq_len, d_v) — 值 value
- mask: (seq_len, seq_len)  — 布尔掩码，True 表示允许注意力，False 表示屏蔽
... 表示任意数量的 batch-like 维度，如注意力头维度。
'''

'\nProblem 6: 实现缩放点积注意力 (5 points)\n要求：实现缩放点积注意力，支持任意 batch 维度和可选的注意力掩码。核心参数有：\n- q, k: (batch_size, ..., seq_len, d_k) — 查询 query 和键 key\n- v: (batch_size, ..., seq_len, d_v) — 值 value\n- mask: (seq_len, seq_len)  — 布尔掩码，True 表示允许注意力，False 表示屏蔽\n... 表示任意数量的 batch-like 维度，如注意力头维度。\n'

In [14]:
def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor = None
) -> torch.Tensor:
    """
    实现缩放点积注意力
    q, k: (batch_size, ..., seq_len, d_k)
    v: (batch_size, ..., seq_len, d_v)
    mask: 可选布尔张量 (seq_len, seq_len)
          mask=True 代表该位置允许参与注意力权重计算, False 权重置零
    return: (batch_size, ..., seq_len, d_v)
    """

    scores=q @ k.transpose(-2,-1)
    scores/=(q.shape[-1]**0.5)  #根号d_k
    
    if mask is not None:
        scores=scores.masked_fill(mask==False,-torch.inf)  # 如果是False，填充负无穷

    scores=softmax(scores,-1)

    y=scores @ v

    return y 

In [15]:
'''
Problem 7: 实现因果多头自注意力 (5 points)
要求：从零实现带因果掩码和 RoPE 的多头自注意力模块。遵循原始 transformer，设置 $$d_{head} = d_k = d_v = d_{model} / h$$。接收的核心参数有：
- d_model: int — Transformer 块输入维度
- num_heads: int — 注意力头数
设计要点：
- QKV 投影矩阵可以合并为一个大矩阵，仅需一次矩阵乘法就能计算出 Q、K、V 向量；
- RoPE 只应用于 Q 和 K，不应用于 V；
- 因果掩码使用下三角布尔矩阵。
'''

'\nProblem 7: 实现因果多头自注意力 (5 points)\n要求：从零实现带因果掩码和 RoPE 的多头自注意力模块。遵循原始 transformer，设置 $$d_{head} = d_k = d_v = d_{model} / h$$。接收的核心参数有：\n- d_model: int — Transformer 块输入维度\n- num_heads: int — 注意力头数\n设计要点：\n- QKV 投影矩阵可以合并为一个大矩阵，仅需一次矩阵乘法就能计算出 Q、K、V 向量；\n- RoPE 只应用于 Q 和 K，不应用于 V；\n- 因果掩码使用下三角布尔矩阵。\n'

In [16]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, max_seq_len: int, rope_theta: float) -> None:
        super().__init__()
        '''
        d_model: int  Transformer 块输入维度
        num_heads: int  注意力头数
        max_seq_len: int  RoPE 最大序列长度
        rope_theta: float  RoPE 基数频率
        '''
        self.d_model=d_model
        assert d_model % num_heads == 0 ,"d_model / num_heads 无法整除"
        self.d_head=d_model // num_heads #计算每个头的向量维度
        self.num_heads=num_heads

        self.w_qkv=Linear(d_model,3*d_model)

        self.rope=Rope(theta=rope_theta,d_k=self.d_head,max_seq_len=max_seq_len)

        self.w_output=Linear(d_model,d_model)


    def forward(self, x: torch.Tensor, token_positions: torch.Tensor = None) -> torch.Tensor:
        '''
        x: (batch_size, seq_len, d_model)  输入张量
        token_positions: (batch_size, seq_len) 或 None  各 token 的位置下标
        返回形状: (batch_size, seq_len, d_model)
        '''
        
        qkv=self.w_qkv(x)

        q,k,v=torch.split(qkv,self.d_model,dim=-1) #(B,S,3*d_model)  -> q/k/v:(B,S,d_model)

        # 拆分多头：把 d_model 拆成 num_heads × d_head，num_heads 放到 seq 前面，匹配 rope 的代码（回顾一下 rope 对输入 x 的要求）
        # (B, S, num_heads*d_head) → (B, num_heads, S, d_head)
        #原始每个token后面挨着所有head的数据 现在每个head都有一整套序列
        q = rearrange(q, "... seq (num_heads d_head) -> ... num_heads seq d_head", num_heads=self.num_heads, d_head=self.d_head)
        k = rearrange(k, "... seq (num_heads d_head) -> ... num_heads seq d_head", num_heads=self.num_heads, d_head=self.d_head)
        v = rearrange(v, "... seq (num_heads d_head) -> ... num_heads seq d_head", num_heads=self.num_heads, d_head=self.d_head)

        seq_len = x.shape[1]

        
        if self.rope is not None: # 是否使用 rope
            if token_positions is None:
                token_positions = torch.arange(seq_len, device=x.device) # 未传入位置时默认连续位置 0,1,2...seq_len‑1
            q = self.rope(q, token_positions)
            k = self.rope(k, token_positions)

        # 使用 tril(lower triangle) 构造下三角矩阵，下三角位置为全 1，其他位置为全 0
        mask = torch.tril(torch.ones((seq_len, seq_len), device=q.device)).bool() # 掩码矩阵 (seq_len, seq_len)：只允许看当前以及历史 token，不能看未来 token

        output_head = scaled_dot_product_attention(q, k, v, mask) # (B, n_head, S, d_head) -> (B, n_head, S, d_head)，B 和 n_head 均为批次维度，执行广播操作

        output_head = rearrange(output_head, "... num_heads seq d_head -> ... seq (num_heads d_head)") # 多头结果拼接: (B, n_heads, S, d_head) → (B, S, n_heads*d_head=d_model)
        
        output = self.w_output(output_head) # 输出层投影: (B, S, d_model) -> (B, S, d_model)
        
        return output




In [17]:
# 测试 scaled_dot_product_attention 带因果掩码
B, S, dk, dv = 2, 4, 8, 8
q = torch.randn(B, S, dk)
k = torch.randn(B, S, dk)
v = torch.randn(B, S, dv)
# 因果下三角掩码
causal_mask = torch.tril(torch.ones(S, S)).bool()
attn_out = scaled_dot_product_attention(q, k, v, mask=causal_mask)
print(f"输入 q/k/v 形状: {q.shape}")
print(f"注意力输出 形状: {attn_out.shape}")

输入 q/k/v 形状: torch.Size([2, 4, 8])
注意力输出 形状: torch.Size([2, 4, 8])
